# J12 — R01 supervised inference on N01 HAM/BCN validation subset

## 1. Frozen N01 validation IDs

In [1]:
import os,io,json,hashlib,subprocess,zipfile,time,sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torchvision.models import resnet50
from torchvision import transforms
from PIL import Image,ImageOps
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
TEMP=Path('/kaggle/temp/j12' if Path('/kaggle/temp').exists() else '/mnt/data/j12_tmp');TEMP.mkdir(parents=True,exist_ok=True)
OUT=WORK/'j12-lesion-aware-supervised-ham-bcn'
for d in ['reports','tables','predictions']:(OUT/d).mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
req_gpu=os.getenv('J12_ALLOW_CPU','0')!='1';
if req_gpu and DEVICE.type!='cuda':raise RuntimeError('Enable Kaggle GPU or set J12_ALLOW_CPU=1')
def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def write(p,x):Path(p).write_text(json.dumps(x,indent=2,default=str))
# Reuse the EXACT G03-B validation IDs, labels and sources (not a newly created split).
found=[]
for p in INPUT.rglob('final_status.json'):
    try:
        x=json.loads(p.read_text())
        if x.get('stage')=='G03-B' and x.get('status')=='COMPLETED':found.append(p)
    except Exception:continue
req(len(found)==1,'Attach exactly one completed G03-B output')
meta_path=found[0].parent.parent/'predictions/ensemble_probabilities.npz'
with np.load(meta_path,allow_pickle=True) as z:
    for k in ['image_id','source','group_id']:
        a=z[k];req(a.ndim==1 and all(isinstance(t,(str,np.str_)) for t in a),f'{k} invalid')
    ids=z['image_id'].astype(str);origin=z['source'].astype(str);group=z['group_id'].astype(str);y=z['y'].astype(int)
req(len(set(ids))==len(ids) and len(ids)==len(y),'G03-B identity mismatch')
print('N01 validation source subset:',len(y),pd.Series(origin).value_counts().to_dict())


N01 validation source subset: 2109 {'HAM10000': 1060, 'BCN20000': 1049}


## 2. R01 checkpoint audit and architecture load

In [2]:
# J12 — replace checkpoint discovery/selection, up to the line `# torch.load loads pickles...`
import re
from pathlib import Path

def select_r01_checkpoint():
    override=os.getenv('J12_R01_CHECKPOINT','').strip()
    if override:
        selected=Path(override)
        req(selected.is_file(),f'J12_R01_CHECKPOINT does not exist: {selected}')
        req(selected.suffix.lower() in {'.pt','.pth'},f'Unsupported checkpoint extension: {selected}')
        print('R01 checkpoint explicitly selected:',selected)
        return selected

    files=sorted({p.resolve() for ext in ('*.pt','*.pth')
                  for p in INPUT.rglob(ext)
                  if p.is_file() and any(t in str(p).lower()
                      for t in ('/r01/','r01-supervised','lesion-r01'))})
    req(bool(files),'No R01 checkpoints found. Attach output from lesion-r01-supervised-resnet50-s42.')
    groups={}
    for p in files:
        digest=sha(p)
        groups.setdefault(digest,[]).append(p)
    unique=sorted((paths[0] for paths in groups.values()),key=str)
    report=[{'path':str(p),'filename':p.name,'size_mb':round(p.stat().st_size/1e6,2),
             'sha256':sha(p),'same_checkpoint_copies':len(groups[sha(p)])} for p in unique]
    write(OUT/'reports/r01_checkpoint_candidates.json',report)
    print('R01 discovered files:',len(files),'unique checkpoints:',len(unique))
    for i,x in enumerate(report):
        print(f"{i}: {x['filename']} ({x['size_mb']} MB) SHA={x['sha256'][:12]} PATH={x['path']}")
    if len(unique)==1:return unique[0]

    best=[p for p in unique if re.search(r'(^|[/_.-])best([/_.-]|$)',str(p).lower())]
    if len(best)==1:
        print('Selected uniquely named BEST-validation checkpoint:',best[0])
        return best[0]

    raise RuntimeError('Multiple distinct R01 checkpoints without a unique BEST checkpoint. '
        'Review reports/r01_checkpoint_candidates.json and set '
        "os.environ['J12_R01_CHECKPOINT'] to the full intended best-validation checkpoint path. "
        'Never select a checkpoint merely because it appears first.')

ckpt=select_r01_checkpoint()

# torch.load loads pickles: use ONLY your own trusted Kaggle R01 output.
blob=torch.load(ckpt,map_location='cpu',weights_only=False)
req(isinstance(blob,dict),'R01 checkpoint is not a dictionary')
state=next((blob[k] for k in ['model','model_state_dict','state_dict','model_weights']
            if k in blob and isinstance(blob[k],dict)),blob)
req(isinstance(state,dict) and state and all(torch.is_tensor(v) for v in state.values()),'Unexpected R01 state dictionary')
def strip_prefix(weights):
    for pre in ['module.','model.','net.']:
        if all(k.startswith(pre) for k in weights):weights={k[len(pre):]:v for k,v in weights.items()}
    return weights
state=strip_prefix(state)
class WrappedResNet(nn.Module):
    def __init__(self):
        super().__init__();self.encoder=resnet50(weights=None);self.encoder.fc=nn.Identity();self.classifier=nn.Linear(2048,8)
    def forward(self,x):return self.classifier(self.encoder(x))
if 'fc.weight' in state and state['fc.weight'].shape==(8,2048):
    model=resnet50(weights=None);model.fc=nn.Linear(2048,8);architecture='torchvision_resnet50_fc'
elif 'classifier.weight' in state and state['classifier.weight'].shape==(8,2048) and 'encoder.conv1.weight' in state:
    model=WrappedResNet();architecture='R04_compatible_encoder_classifier'
else:
    write(OUT/'reports/r01_unrecognized_architecture.json',{'checkpoint':str(ckpt),'state_keys':list(state)[:100],
         'tensor_shapes':{k:list(v.shape) for k,v in list(state.items())[:100]}})
    raise RuntimeError('R01 checkpoint architecture cannot be established from saved tensors; see r01_unrecognized_architecture.json. The original R01 architecture source is required.')
model.load_state_dict(state,strict=True);model.to(DEVICE).eval()
write(OUT/'reports/trained_parameter_counts.json',{'Supervised_R01':sum(p.numel() for p in model.parameters())/1e6})
print('Loaded original R01 architecture:',architecture,'model parameters:',sum(p.numel() for p in model.parameters()))


R01 discovered files: 2 unique checkpoints: 2
0: best.pt (94.4 MB) SHA=ce0af27a0ff6 PATH=/kaggle/input/notebooks/sudiptoroy007/lesion-r01-supervised-resnet50-s42/lesion-r01-supervised-resnet50-s42/models/best.pt
1: last.pt (94.42 MB) SHA=b8db5f205537 PATH=/kaggle/input/notebooks/sudiptoroy007/lesion-r01-supervised-resnet50-s42/lesion-r01-supervised-resnet50-s42/training_state/last.pt
Selected uniquely named BEST-validation checkpoint: /kaggle/input/notebooks/sudiptoroy007/lesion-r01-supervised-resnet50-s42/lesion-r01-supervised-resnet50-s42/models/best.pt
Loaded original R01 architecture: R04_compatible_encoder_classifier model parameters: 23524424


## 3. Exact N01 preprocessing and evaluation

In [3]:
# Use N01 exact prepared JPEGs if attached, otherwise reconstruct from the official ISIC2019 ZIP.
manifest_candidates=sorted(INPUT.rglob('experiment_master_manifest.csv'));prepared={}
for p in manifest_candidates:
    root=p.parent.parent
    if not (root/'images').is_dir():continue
    m=pd.read_csv(p,dtype=str,keep_default_na=False)
    if not {'image_id','image_relpath','resized_sha256'}.issubset(m.columns):continue
    look=m.set_index('image_id');paths={}
    for ident in ids:
        if ident not in look.index:break
        rel=Path(look.loc[ident,'image_relpath']);req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe N01 image_relpath')
        file=root/rel
        if not file.is_file():break
        paths[ident]=file
    if len(paths)==len(ids):
        req(not prepared,'More than one complete N01 image root; attach only one')
        prepared=paths
norm=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
def prepared_bytes(source):
    with Image.open(io.BytesIO(source)) as im:
        im.load();rgb=ImageOps.exif_transpose(im).convert('RGB')
    t=np.asarray(rgb.resize((32,32),Image.Resampling.BILINEAR))
    edges=np.concatenate([t[0],t[-1],t[:,0],t[:,-1]],axis=0)
    color=tuple(np.median(edges,axis=0).astype(np.uint8).tolist())
    output=ImageOps.pad(rgb,(224,224),method=Image.Resampling.BICUBIC,color=color)
    buf=io.BytesIO();output.save(buf,format='JPEG',quality=94,subsampling=0,optimize=True)
    return buf.getvalue()
archive=None;members={}
if not prepared:
    archives=[p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file()]
    req(len(archives)<=1,'Multiple official ISIC ZIP archives attached')
    archive=archives[0] if archives else TEMP/'ISIC_2019_Training_Input.zip'
    if not archive.is_file():
        print('Downloading original ISIC2019 ZIP to reconstruct N01 processed images...')
        r=subprocess.run(['curl','-fL','--retry','6','--connect-timeout','40','-o',str(archive),
            'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip'],check=False)
        req(r.returncode==0 and archive.stat().st_size>5*10**9,'Official ISIC image download failed; enable Kaggle Internet')
    with zipfile.ZipFile(archive) as z:
        for f in z.namelist():
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                k=Path(f).stem;req(k not in members,f'Duplicate official image stem {k}');members[k]=f
    req(set(ids).issubset(members),'N01 validation IDs not present in official ZIP')

def decode_bytes(data):
    with Image.open(io.BytesIO(data)) as im:
        im.load();req(im.size==(224,224),'N01 image not 224px')
        return norm(im.convert('RGB'))
BATCH=int(os.getenv('J12_BATCH','24'))
results=[]
with (zipfile.ZipFile(archive) if archive is not None else __import__('contextlib').nullcontext()) as z:
    for off in range(0,len(ids),BATCH):
        batch_ids=ids[off:off+BATCH]
        raw=[prepared[k].read_bytes() if prepared else prepared_bytes(z.read(members[k])) for k in batch_ids]
        x=torch.stack([decode_bytes(t) for t in raw]).to(DEVICE)
        with torch.inference_mode():pr=torch.softmax(model(x).float(),dim=1).cpu().numpy()
        results.append(pr)
        if off%384==0:print('R01 inference',off,'/',len(ids))
p=np.vstack(results);req(p.shape==(len(ids),8) and np.max(abs(p.sum(1)-1))<1e-4,'Invalid R01 outputs')
np.savez_compressed(OUT/'predictions/j12_supervised_predictions.npz',image_id=np.asarray(ids,dtype=str),
    source=np.asarray(origin,dtype=str),group_id=np.asarray(group,dtype=str),y=np.asarray(y,dtype=np.int32),
    proba=p.astype(np.float32),classes=np.asarray(CLASSES),checkpoint_sha256=np.array(sha(ckpt)))
summary=[]
for name in ['HAM10000','BCN20000']:
    mask=origin==name;yt=y[mask];pr=p[mask].argmax(1)
    summary.append({'source':name,'n':int(mask.sum()),'BACC':balanced_accuracy_score(yt,pr),
                    'accuracy':accuracy_score(yt,pr),'MacroF1':f1_score(yt,pr,labels=np.unique(yt),average='macro',zero_division=0)})
pd.DataFrame(summary).to_csv(OUT/'tables/r01_ham_bcn.csv',index=False)
write(OUT/'reports/final_status.json',{'stage':'J12','status':'COMPLETED','architecture':architecture,
    'r01_checkpoint_sha256':sha(ckpt),'n_images':len(ids),'exact_n01_prepared':bool(prepared),'evaluation':'N01_validation_only'})
print(pd.DataFrame(summary).to_string(index=False))


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9318M  100 9318M    0     0  38.3M      0  0:04:02  0:04:02 --:--:-- 76.0M


R01 inference 0 / 2109
R01 inference 384 / 2109
R01 inference 768 / 2109
R01 inference 1152 / 2109
R01 inference 1536 / 2109
R01 inference 1920 / 2109
  source    n     BACC  accuracy  MacroF1
HAM10000 1060 0.684285  0.823585 0.657412
BCN20000 1049 0.465106  0.627264 0.476642
